In [ ]:
#pip install google_play_scraper
#https://pypi.org/project/google-play-scraper/
from google_play_scraper import Sort, reviews
import pandas as pd
import csv


In [ ]:

def fetch_reviews(app_id, lang='en', country='my', max_reviews=1000, max_iterations=300):
    """
    Fetch reviews from Google Play for a specific app.

    Parameters:
        app_id (str): The package name of the app.
        lang (str): Language of reviews (default: 'en').
        country (str): Country code (default: 'my').
        max_reviews (int): Maximum number of reviews to fetch.
        max_iterations (int): Safety limit to avoid infinite loops.

    Returns:
        list[dict]: List of review dictionaries.
    """
    all_reviews = []
    unique_ids = set()
    i = 0

    result, continuation_token = reviews(
        app_id,
        lang=lang,
        country=country,
        sort=Sort.NEWEST,
        count=199,
        filter_score_with=None
    )

    all_reviews.extend(result)
    unique_ids.update(r['reviewId'] for r in result)

    while len(all_reviews) < max_reviews and continuation_token and i < max_iterations:
        next_batch, continuation_token = reviews(app_id, continuation_token=continuation_token)
        new_reviews = [r for r in next_batch if r['reviewId'] not in unique_ids]
        all_reviews.extend(new_reviews)
        unique_ids.update(r['reviewId'] for r in new_reviews)
        i += 1
        print(f"Fetched {len(all_reviews)} reviews...")

    print("Data collection completed!")
    return all_reviews


In [ ]:

def extract_keywords(reviews_list, fields=None):
    """
    Extract selected fields from reviews.

    Parameters:
        reviews_list (list): List of review dictionaries.
        fields (list): List of fields to extract.

    Returns:
        list[dict]: Filtered review dictionaries.
    """
    if fields is None:
        fields = ['reviewId', 'userName', 'userImage', 'content', 'score',
                  'thumbsUpCount', 'reviewCreatedVersion', 'at', 
                  'replyContent', 'repliedAt', 'appVersion']

    return [{key: review.get(key, None) for key in fields} for review in reviews_list]


In [ ]:

def save_reviews_to_csv(reviews_data, filename='app_reviews.csv'):
    """
    Save review data to a CSV file.

    Parameters:
        reviews_data (list): List of dictionaries (filtered reviews).
        filename (str): Output filename.
    """
    if not reviews_data:
        print("No reviews to save.")
        return

    keys = reviews_data[0].keys()
    with open(filename, 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=keys)
        writer.writeheader()
        writer.writerows(reviews_data)

    print(f"Reviews saved to {filename}")


In [ ]:

def read_csv_to_dataframe(filepath):
    """
    Read a CSV file into a pandas DataFrame and return it.

    Parameters:
        filepath (str): Path to the CSV file.

    Returns:
        pd.DataFrame: Loaded DataFrame.
    """
    df = pd.read_csv(filepath)
    print("First 5 rows of the dataset:")
    print(df.head())
    return df


In [ ]:
#https://play.google.com/store/apps/

# Step 1: Fetch
raw_reviews = fetch_reviews('com.maxis.mymaxis', lang='en', country='my', max_reviews=5000, max_iterations=1000)

# Step 2: Filter important fields
filtered_reviews = extract_keywords(raw_reviews)

# Step 3: Save to CSV
save_reviews_to_csv(filtered_reviews, 'Maxis_reviews.csv')

# Step 4: Read into DataFrame
df = read_csv_to_dataframe('Maxis_reviews.csv')


In [ ]:
# Obtain the location of the file
import os
os.path.abspath('.')